In [15]:
%%writefile program.cu
#include<iostream>
using namespace std;

__global__ void vecAdd(float* A,float* B,float *C,int n)
{
  int i=blockIdx.x*blockDim.x+threadIdx.x;
  if(i<n)
  {
    C[i]=A[i]+B[i];
  }
}

__global__ void matMul(float* A,float* B,float* C,int n)
{
  int row=blockIdx.y*blockDim.y+threadIdx.y;
  int col=blockIdx.x*blockDim.x+threadIdx.x;
  if(row<n && col<n)
  {
    float sum=0;
    for(int k=0;k<n;k++)
    {
      sum+=A[row*n+k]*B[k*n+col];
    }
    C[row*n+col]=sum;
  }
}

int main()
{
  float A[]={1,2,3,4};
  float B[]={5,6,7,8};
  float C[4];
  int n=4;

  size_t size=n*sizeof(float);
  float *d_A,*d_B,*d_C;
  cudaMalloc(&d_A,size);
  cudaMalloc(&d_B,size);
  cudaMalloc(&d_C,size);

  cudaMemcpy(d_A,A,size,cudaMemcpyHostToDevice);
  cudaMemcpy(d_B,B,size,cudaMemcpyHostToDevice);

  vecAdd<<<1,n>>>(d_A,d_B,d_C,n);

  cudaMemcpy(C,d_C,size,cudaMemcpyDeviceToHost);
  for(int i=0;i<n;i++)
  {
    cout<<C[i]<<" ";
  }

  int N=2;
  size=N*N*sizeof(float);
  float *d_matA,*d_matB,*d_matC;
  cudaMalloc(&d_matA,size);
  cudaMalloc(&d_matB,size);
  cudaMalloc(&d_matC,size);

  cudaMemcpy(d_matA,A,size,cudaMemcpyHostToDevice);
  cudaMemcpy(d_matB,B,size,cudaMemcpyHostToDevice);

  dim3 threads(2,2);
  dim3 blocks(1,1);

  matMul<<<blocks,threads>>>(d_matA,d_matB,d_matC,N);

  cudaMemcpy(C,d_matC,size,cudaMemcpyDeviceToHost);
  for(int i=0;i<N*N;i++)
  {
    cout<<C[i]<<" ";
  }
  return 0;
}


Overwriting program.cu


In [16]:
!nvcc program.cu -o program

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [17]:
!./program

6 8 10 12 19 22 43 50 